# First-Order and Adaptive Optimisers (C04 Topic 01)

**Citations:** Robbins & Monro (1951 Stochastic Approx); Polyak (1964 Momentum); Duchi et al. (2011 Adagrad); Kingma & Ba (2014 Adam); Loshchilov & Hutter (2017 AdamW); Zhuang et al. (2020 AdaBelief).

### Purpose & Topic Bridges
Non-convex optimization landscapes present ill-conditioned curvatures, ravines, and gradient noise. First-order methods traverse these surfaces using velocity inertia (Polyak Momentum) and per-parameter adaptive gradient scaling (Adagrad, RMSprop, Adam, AdamW, AdaBelief). Decoupled weight decay (AdamW) corrects the fundamental mathematical flaw of coupled $L_2$ regularization in Adam, where adaptive variance division artificially suppresses weight decay on active parameters.

**Decision Boundary:** Use **AdamW** as the universal default for Transformers, LLMs, and multi-modal models. Use **SGD with Momentum** for training standard CNNs from scratch on vision benchmarks where implicit flat-minima regularization yields superior out-of-distribution generalization. Never use coupled $L_2$ decay in standard Adam when genuine weight shrinkage is required.

## 2. Mathematical Foundations & Governing Equations

1. **SGD with Momentum:** $v_{t+1} = \mu v_t + (g_t + \lambda \theta_t), \quad \theta_{t+1} = \theta_t - \eta v_{t+1}$.
2. **Adagrad:** $G_{t+1} = G_t + g_t^2, \quad \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_{t+1}} + \epsilon} g_t$.
3. **RMSprop:** $v_{t+1} = \alpha v_t + (1 - \alpha) g_t^2, \quad \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_{t+1}} + \epsilon} g_t$.
4. **Adam (Coupled $L_2$):** $\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$, where $\tilde{g}_t = g_t + \lambda \theta_t$.
5. **AdamW (Decoupled Weight Decay):** $\theta_{t+1} = \theta_t (1 - \eta \lambda) - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$, with pure $g_t$ driving $m_t$ and $v_t$.
6. **AdaBelief:** Tracks gradient variance relative to belief: $s_{t+1} = \beta_2 s_t + (1 - \beta_2)(g_t - m_{t+1})^2 + \epsilon$.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/01_first_order_and_adaptive_optimisers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class FirstOrderOptimisersUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=8):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def step_sgd_momentum(self, p, g, v, lr=0.05, mom=0.9, wd=0.01):
        grad = g + wd * p; new_v = mom * v + grad; return p - lr * new_v, new_v

    def step_adagrad(self, p, g, acc, lr=0.05, eps=1e-8, wd=0.01):
        grad = g + wd * p; new_acc = acc + grad * grad; return p - lr * grad / (torch.sqrt(new_acc) + eps), new_acc

    def step_rmsprop(self, p, g, v, lr=0.01, alpha=0.99, eps=1e-8, wd=0.01):
        grad = g + wd * p; new_v = alpha * v + (1.0 - alpha) * grad * grad; return p - lr * grad / (torch.sqrt(new_v) + eps), new_v

    def step_adam(self, p, g, m, v, step=2, lr=0.001, b1=0.9, b2=0.999, eps=1e-8, wd=0.01):
        grad = g + wd * p; new_m = b1 * m + (1.0 - b1) * grad; new_v = b2 * v + (1.0 - b2) * grad * grad
        m_hat = new_m / (1.0 - b1**step); v_hat = new_v / (1.0 - b2**step)
        return p - lr * m_hat / (torch.sqrt(v_hat) + eps), new_m, new_v

    def step_adamw(self, p, g, m, v, step=2, lr=0.001, b1=0.9, b2=0.999, eps=1e-8, wd=0.01):
        new_m = b1 * m + (1.0 - b1) * g; new_v = b2 * v + (1.0 - b2) * g * g
        m_hat = new_m / (1.0 - b1**step); v_hat = new_v / (1.0 - b2**step)
        return p * (1.0 - lr * wd) - lr * m_hat / (torch.sqrt(v_hat) + eps), new_m, new_v

    def step_adabelief(self, p, g, m, s, step=2, lr=0.001, b1=0.9, b2=0.999, eps=1e-16, wd=0.01):
        new_m = b1 * m + (1.0 - b1) * g; diff = g - new_m; new_s = b2 * s + (1.0 - b2) * diff * diff + eps
        m_hat = new_m / (1.0 - b1**step); s_hat = new_s / (1.0 - b2**step)
        return p * (1.0 - lr * wd) - lr * m_hat / (torch.sqrt(s_hat) + eps), new_m, new_s

model = FirstOrderOptimisersUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['core_weight'].to(device))
    model.linear.bias.copy_(weights['core_bias'].to(device))
print('FirstOrderOptimisersUnifiedModule initialized successfully.')

In [ ]:
p_in = weights['param'].to(device); g_in = weights['grad'].to(device)
m_in = weights['m_init'].to(device); v_in = weights['v_init'].to(device)

p_sgd, v_sgd = model.step_sgd_momentum(p_in, g_in, m_in)
err_sgd = max((p_sgd - weights['p_sgd'].to(device)).abs().max().item(), (v_sgd - weights['v_sgd'].to(device)).abs().max().item())

p_ada, g_ada = model.step_adagrad(p_in, g_in, v_in)
err_ada = max((p_ada - weights['p_adagrad'].to(device)).abs().max().item(), (g_ada - weights['g_adagrad'].to(device)).abs().max().item())

p_rms, v_rms = model.step_rmsprop(p_in, g_in, v_in)
err_rms = max((p_rms - weights['p_rmsprop'].to(device)).abs().max().item(), (v_rms - weights['v_rmsprop'].to(device)).abs().max().item())

p_adam, m_adam, v_adam = model.step_adam(p_in, g_in, m_in, v_in)
err_adam = max((p_adam - weights['p_adam'].to(device)).abs().max().item(), (m_adam - weights['m_adam'].to(device)).abs().max().item(), (v_adam - weights['v_adam'].to(device)).abs().max().item())

p_adamw, m_adamw, v_adamw = model.step_adamw(p_in, g_in, m_in, v_in)
err_adamw = max((p_adamw - weights['p_adamw'].to(device)).abs().max().item(), (m_adamw - weights['m_adamw'].to(device)).abs().max().item(), (v_adamw - weights['v_adamw'].to(device)).abs().max().item())

p_adab, m_adab, s_adab = model.step_adabelief(p_in, g_in, m_in, v_in)
err_adab = max((p_adab - weights['p_adabelief'].to(device)).abs().max().item(), (m_adab - weights['m_adabelief'].to(device)).abs().max().item(), (s_adab - weights['s_adabelief'].to(device)).abs().max().item())

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor error: {err_anchor:.2e}'
print(f'All 7 invariants verified! SGD: {err_sgd:.2e}, Ada: {err_ada:.2e}, RMS: {err_rms:.2e}, Adam: {err_adam:.2e}, AdamW: {err_adamw:.2e}, AdaBelief: {err_adab:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'SGD Momentum': lambda: model.step_sgd_momentum(p_in, g_in, m_in),
    'Adagrad': lambda: model.step_adagrad(p_in, g_in, v_in),
    'RMSprop': lambda: model.step_rmsprop(p_in, g_in, v_in),
    'Adam (Coupled)': lambda: model.step_adam(p_in, g_in, m_in, v_in),
    'AdamW (Decoupled)': lambda: model.step_adamw(p_in, g_in, m_in, v_in),
    'AdaBelief': lambda: model.step_adabelief(p_in, g_in, m_in, v_in),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Coupled L2 Decay Suppression in Standard Adam vs Decoupled AdamW
# When parameters have large gradients, Adam scales weight decay by 1 / sqrt(v_t), suppressing decay!
# On frequently updated weights (large grad |g| = 10.0), Adam fails to shrink weights effectively:
param_test = torch.tensor([5.0], device=device); grad_large = torch.tensor([10.0], device=device)
m_zero = torch.tensor([0.0], device=device); v_zero = torch.tensor([0.0], device=device)
lr = 0.01; wd = 0.1

p_clean, _, _ = model.step_adamw(param_test, grad_large, m_zero, v_zero, step=1, lr=lr, wd=0.0)
p_adam_out, _, _ = model.step_adam(param_test, grad_large, m_zero, v_zero, step=1, lr=lr, wd=wd)
p_adamw_out, _, _ = model.step_adamw(param_test, grad_large, m_zero, v_zero, step=1, lr=lr, wd=wd)

# Adam decay effect is scaled down to 0 due to gradient normalization, while AdamW shrinks exactly:
adam_decay = float(p_clean - p_adam_out)
adamw_decay = float(p_clean - p_adamw_out)
print(f'[FAILURE DEMONSTRATED] Adam suppressed L2 decay: {adam_decay:.6f} vs AdamW true weight decay: {adamw_decay:.6f}')
assert abs(adam_decay) < 1e-4, 'Adam failed to suppress coupled decay on large gradient'
assert abs(adamw_decay - (param_test.item() * lr * wd)) < 1e-5, 'AdamW did not apply exact proportional decay'
print('[CORRECT] Decoupled AdamW preserves invariant weight shrinkage regardless of gradient magnitude.')


## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Decoupled Weight Decay is Mandatory:** Adam couples $L_2$ decay to variance preconditioning, suppressing regularization on active parameters. AdamW restores scale-invariant shrinkage $(1 - \eta \lambda)$.
2. **State Memory Trade-Off:** Adaptive methods require $2\times$ parameter state storage ($m_t, v_t$). At scale, fused GPU kernels and factored optimisers (Adafactor/CAME) mitigate bandwidth bottlenecks.
3. **Physical & Control Isomorphisms:** Momentum acts as inertial mass damping harmonic oscillations, while variance tracking implements Automatic Gain Control (AGC) over non-stationary loss gradients.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 03_building_blocks_and_losses/13_differentiable_sorting_ranking_topk_permutahedron_and_imle](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/13_differentiable_sorting_ranking_topk_permutahedron_and_imle.ipynb)